In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.decomposition import PCA

from Src.data import load_csv_df
from Src.plotting import set_plotting_style

In [5]:
# Already prepared accent pallette applying using function from mine Src module (plotting)
set_plotting_style()

In [6]:
DF_PATH = "../Data/Raw/raw-data.csv"

# Safely loading dataframe using function from mine Src module (data)
df = load_csv_df(
    df_path=DF_PATH,
)

Successfully loaded dataframe: ..\Data\Raw\raw-data.csv
Dataframe has: 8950 rows / 18 columns
Dataframe size: 0.861MB


In [7]:
# Dropping CUST_ID because it has a lot of unique values (See in EDA)
df = df.drop(columns=["CUST_ID"])

In [8]:
# Filling 314 NaN Total with medians (Read more in EDA)
df["CREDIT_LIMIT"] = df["CREDIT_LIMIT"].fillna(df["CREDIT_LIMIT"].median())
df["MINIMUM_PAYMENTS"] = df["MINIMUM_PAYMENTS"].fillna(df["MINIMUM_PAYMENTS"].median())

In [9]:
NUMERIC_COL = df.select_dtypes(exclude=["object", "string", "category"]).columns

# Why these columns (See in EDA hist plots analysis)
EXCLUDE_COLS = [
    "TENURE",
    "PURCHASES_FREQUENCY",
    "BALANCE_FREQUENCY",
    "ONEOFF_PURCHASES_FREQUENCY",
    "CASH_ADVANCE_FREQUENCY"
    "PURCHASES_INSTALLMENTS_FREQUENCY",
    "PRC_FULL_PAYMENT",
]
TARGET_COLS = NUMERIC_COL.difference(EXCLUDE_COLS, sort=False).tolist()
print(TARGET_COLS)

['BALANCE', 'BALANCE_FREQUENCY', 'PURCHASES', 'ONEOFF_PURCHASES', 'INSTALLMENTS_PURCHASES', 'CASH_ADVANCE', 'ONEOFF_PURCHASES_FREQUENCY', 'PURCHASES_INSTALLMENTS_FREQUENCY', 'CASH_ADVANCE_FREQUENCY', 'CASH_ADVANCE_TRX', 'PURCHASES_TRX', 'CREDIT_LIMIT', 'PAYMENTS', 'MINIMUM_PAYMENTS']


In [ ]:
# Log1p to remove all skews and outliers
preprocessor = ColumnTransformer(
    transformers=[
        ("log1p", FunctionTransformer, TARGET_COLS),
        ("passthrough", "passthrough", EXCLUDE_COLS)
    ]
)
x_log = preprocessor.fit_transform(df)